## 8.2 Content

Compile MC.cu using

In [1]:
!nvcc MC.cu -o MC

MC.cu
tmpxft_0000ff74_00000000-10_MC.cudafe1.cpp
   Creating library MC.lib and object MC.exp


Execute MC using (on Microsoft Windows OS ./ is not needed)

In [4]:
!MC

The estimated price is equal to 0.000000
error associated to a confidence interval of 95% = 0.000000
The true price 6.634843
Execution time 0.069664 ms


As long as you did not include any additional instruction in the file MC.cu, the execution above is to return 0. At least, no compilation error is detected by the compiler!


### 8.2.1 (Pseudo) Random number generators, `init_curand_state_k`

Using cuRAND documentation, we need to find a good RNG and understand how it can be used for our simulation.

a) Which of HOST API or DEVICE API should we use for the random numbers generation? 

**Answer**: Use the DEVICE API when random numbers must be generated inside custom CUDA kernels, typically with one RNG state per thread. The HOST API is appropriate for bulk generation of random numbers from the CPU side into device memory. See a complete description [here](https://docs.nvidia.com/cuda/curand/introduction.html#introduction). In this lab, we use the DEVICE API.

b) Print the size of various state options (`XORWOW`, `Mtgp32_t`, and others) and explain why some of them must not be used. 

**Answer**: Some generators (e.g. MTGP32) have large states and are not designed for one-state-per-thread usage. Large states increase register pressure, may spill into local memory, and reduce occupancy. Therefore they are unsuitable for massively parallel per-thread RNG.

c) Which RNG should you use by default? Explain how its initialization is performed. 

**Answer**: Historically, cuda runs the XORWOW algorithm by default due to its small state and high speed. Only XOR and SHIFT operations are performed, and these are extremely fast on ALUs by design. Also, it is easy to parallelize by threads, and the state vector is not lightweight, only $160+32$ bits. One can also mention *Philox4x32-10* and *MRG32k3a* PRNG, the former being more and more predominant in modern HPC on GPUs.

d) Complete the syntax of init_curand_state_k

**Answer**: See `MC.cu` file.


### 8.2.2 Monte Carlo simulation $\text{MC}_{k1}$, with a reduction on the host

`sum` and `sum2` are two variables that should respectively contain the estimation of the first and the second moment of the actualized payoff: $\exp(-rT)\times (x-K)_+$. Their computation with an average is performed on the host. Thus, one needs only to simulate the realizations of the actualized payoff on the device.

a) Define the arrays of the actualized payoffs and perform the right copy.

b) Inspired by examples from cuRAND documentation, define the kernel $\text{MC}_{k1}$.

c) In which situation should we copy back the state to the global memory before exiting the kernel $\text{MC}_{k1}$? 

**Answer**: Copying the state to a global variable is necessary if you intend to execute multiple Monte-Carlo simulations in a row. By copying back the state to global memory, you can change the seed of the PRNG at every MC run.

d) See how the execution time changes with respect to $n$. What do you conclude about the scalability of Monte Carlo simulation?

**Answer**: 
| NB      | NTPB | Execution time (ms)     |
| :---        |    :----:   |          ---: |
| 512      | 512       | 0.65   |
| 1024   | 512        | 0.71      |
| 512   | 1024        | 0.84      |
| 1024   | 1024        | 1.6      |
| 2048   | 512        |  1.58     |
| 512   | 2048        |  Error     |
| 2048   | 1024        |  5     |
| 127   | 1024        |  0.43     |

Notice that an error appears because NTPB exceeds the block capacity `blockDim.x`.